# AIO Presence & Overlap Analysis — Part 3: Entity-Aware Overlap
Resolves AIO/organic citations to canonical media entities and recomputes overlap.
Standalone-runnable — loads and cleans its own data (and rebuilds YouTube channel resolution via the existing §9 bootstrap cell, since df_yt originates in Part 2).

## Setup — load & clean data (shared preamble, standalone-runnable)

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from common import (
    load_data,
    set_plot_style,
    PALETTE,
)

sns.set_theme(style="whitegrid")
set_plot_style()
df = load_data()

## 9. Entity-Aware AIO–Organic Overlap

The standard `aio_organic_overlap` metric matches on **domain strings** only.
This creates two distortions:

1. **Spurious matches** — AIO cites *any* YouTube video, and a different YouTube
   video happens to rank in organic → `youtube.com` matches `youtube.com`.
   In this dataset **23% of AIO queries** have this pattern.
   It is not a real overlap: the two YouTube citations are entirely different
   channels / videos.

2. **Missed matches** — AIO cites the La7 Attualità YouTube channel, while
   `la7.it` ranks in organic. The same brand is cited twice but the domain-level
   metric misses it.

This section resolves both AIO citations (domains + YouTube channels) and organic
domains to a **canonical media entity** and recomputes overlap at that level.

> Depends on `df_yt` produced in §6e (YouTube channel resolution).
> Run §6e first; if skipped, YouTube→entity matching is omitted gracefully.

In [ ]:
# ── §9 bootstrap: reload data + YouTube channels if running standalone ──────
import json as _json
import re as _re
import sys as _sys
from pathlib import Path as _Path
from urllib.parse import urlparse as _urlparse, parse_qs as _parse_qs

import requests as _req
import pandas as _pd

_ROOT = _Path("..").resolve()
_RAW_DIR = _ROOT / "data" / "raw"

_sys.path.insert(0, str(_ROOT))
from config import YOUTUBE_API_KEY as _YT_KEY  # noqa: E402

# ── reload df if not already in scope ───────────────────────────────────────
if "df" not in dir() or df is None:
    print("Loading raw data…")
    from common import load_data

    df = load_data()
else:
    print("df already in scope — skipping reload")


# ── rebuild df_yt if not already in scope ───────────────────────────────────
def _parse_yt_url(url):
    """Return (yt_type, yt_name_or_vid_id) for a YouTube URL."""
    try:
        p = _urlparse(url)
        path = p.path.rstrip("/")
    except Exception:
        return "invalid", None
    for pat, typ in [
        (r"^/@([^/]+)", "handle"),
        (r"^/user/([^/]+)", "username"),
        (r"^/c/([^/]+)", "custom"),
    ]:
        m = _re.match(pat, path)
        if m:
            return typ, m.group(1)
    m = _re.match(r"^/channel/(UC[A-Za-z0-9_-]+)", path)
    if m:
        return "channel_id", m.group(1)
    m = _re.match(r"^/shorts/([A-Za-z0-9_-]+)", path)
    if m:
        return "short", m.group(1)
    qs = _parse_qs(p.query)
    if "v" in qs:
        return "video", qs["v"][0]
    return "other", None


def _yt_api_batch(endpoint, id_param, ids, key):
    results = {}
    for i in range(0, len(ids), 50):
        batch = [x for x in ids[i : i + 50] if x]
        if not batch:
            continue
        try:
            r = _req.get(
                f"https://www.googleapis.com/youtube/v3/{endpoint}",
                params={"part": "snippet", id_param: ",".join(batch), "key": key},
                timeout=15,
            )
            r.raise_for_status()
            for item in r.json().get("items", []):
                results[item["id"]] = item.get("snippet", {})
        except Exception as e:
            print(f"  YouTube API error: {e}")
    return results


if "df_yt" not in dir() or df_yt is None or df_yt.empty:  # noqa: F821
    print("Resolving YouTube channels…")
    _df_aio = df[df["has_ai_overview"]]
    _yt_rows = []
    for _, _row in _df_aio.iterrows():
        _raw = _row.get("aio_sources")
        if not _raw or isinstance(_raw, float):
            continue
        try:
            _srcs = _json.loads(_raw) if isinstance(_raw, str) else _raw
        except Exception:
            continue
        for _src in _srcs:
            _link = _src.get("link", "")
            _title = _src.get("title", "")
            if "youtube.com" not in _link and "youtu.be" not in _link:
                continue
            _typ, _val = _parse_yt_url(_link)
            _yt_rows.append(
                {
                    "query": _row["query"],
                    "topic": _row.get("topic"),
                    "stance": _row.get("stance"),
                    "pro_leaning": _row.get("pro_leaning"),
                    "title": _title,
                    "link": _link,
                    "yt_type": _typ,
                    "yt_name": _val
                    if _typ in ("handle", "username", "custom")
                    else None,
                    "video_id": _val if _typ in ("video", "short") else None,
                    "channel_id_raw": _val if _typ == "channel_id" else None,
                }
            )
    df_yt = (
        _pd.DataFrame(_yt_rows)
        .drop_duplicates(["query", "link"])
        .reset_index(drop=True)
    )
    if not df_yt.empty and _YT_KEY:
        _vid_ids = df_yt.loc[df_yt["video_id"].notna(), "video_id"].unique().tolist()
        _chan_ids = (
            df_yt.loc[df_yt["channel_id_raw"].notna(), "channel_id_raw"]
            .unique()
            .tolist()
        )
        _vid_info = _yt_api_batch("videos", "id", _vid_ids, _YT_KEY) if _vid_ids else {}
        _chan_info = (
            _yt_api_batch("channels", "id", _chan_ids, _YT_KEY) if _chan_ids else {}
        )

        def _enrich(row):
            if row["video_id"] and row["video_id"] in _vid_info:
                s = _vid_info[row["video_id"]]
                return _pd.Series(
                    {
                        "channel_title": s.get("channelTitle"),
                        "channel_id": s.get("channelId"),
                        "api_resolved": True,
                    }
                )
            if row["channel_id_raw"] and row["channel_id_raw"] in _chan_info:
                s = _chan_info[row["channel_id_raw"]]
                return _pd.Series(
                    {
                        "channel_title": s.get("title"),
                        "channel_id": row["channel_id_raw"],
                        "api_resolved": True,
                    }
                )
            return _pd.Series(
                {
                    "channel_title": row["yt_name"],
                    "channel_id": row.get("channel_id_raw"),
                    "api_resolved": False,
                }
            )

        df_yt[["channel_title", "channel_id", "api_resolved"]] = df_yt.apply(
            _enrich, axis=1
        )
    else:
        df_yt["channel_title"] = df_yt.get("yt_name")
        df_yt["channel_id"] = df_yt.get("channel_id_raw")
        df_yt["api_resolved"] = False
    df_yt["channel_label"] = df_yt["channel_title"].fillna("(unknown)")
    print(
        f"  df_yt ready: {len(df_yt)} citations, "
        f"{df_yt['channel_label'].nunique()} unique channel labels"
    )
else:
    print("df_yt already in scope — skipping YouTube resolution")

In [ ]:
# ── Canonical entity map ────────────────────────────────────────────────────
# Keys are lowercase domain names OR lowercase YouTube channel names.
# Values are the canonical entity name used in the overlap calculation.
# Extend this dict whenever you add new data.

ENTITY_MAP = {
    # ── La7 ──────────────────────────────────────────────────────────────
    "la7.it": "La7",
    "la7 attualità": "La7",
    "la7attualità": "La7",
    "tg la7": "La7",
    "tg la7 attualità": "La7",
    "la7": "La7",
    # ── Il Fatto Quotidiano ───────────────────────────────────────────────
    "ilfattoquotidiano.it": "Il Fatto Quotidiano",
    "il fatto quotidiano": "Il Fatto Quotidiano",
    "fatto quotidiano": "Il Fatto Quotidiano",
    # ── RAI ──────────────────────────────────────────────────────────────
    "rai.it": "RAI",
    "rainews.it": "RAI",
    "rainews": "RAI",
    "rai news": "RAI",
    "rai news 24": "RAI",
    "rai": "RAI",
    # ── Il Sole 24 ORE ────────────────────────────────────────────────────
    "ilsole24ore.com": "Il Sole 24 ORE",
    "il sole 24 ore": "Il Sole 24 ORE",
    "sole 24 ore": "Il Sole 24 ORE",
    # ── Altalex ──────────────────────────────────────────────────────────
    "altalex.com": "Altalex",
    "altalex news": "Altalex",
    "altalex": "Altalex",
    # ── Sky TG24 ─────────────────────────────────────────────────────────
    "tg24.sky.it": "Sky TG24",
    "sky tg24": "Sky TG24",
    "skytg24": "Sky TG24",
    # ── L'Espresso ───────────────────────────────────────────────────────
    "lespresso.it": "L'Espresso",
    "l'espresso": "L'Espresso",
    "espresso": "L'Espresso",
    # ── Vatican News ─────────────────────────────────────────────────────
    "vaticannews.va": "Vatican News",
    "vatican news - italiano": "Vatican News",
    "vatican news": "Vatican News",
    # ── Euronews ─────────────────────────────────────────────────────────
    "it.euronews.com": "Euronews",
    "euronews.com": "Euronews",
    "euronews (in italiano)": "Euronews",
    "euronews italiano": "Euronews",
    # ── Avvenire ─────────────────────────────────────────────────────────
    "avvenire.it": "Avvenire",
    "avvenire": "Avvenire",
    # ── ANSA ─────────────────────────────────────────────────────────────
    "ansa.it": "ANSA",
    "ansa": "ANSA",
    "agensir": "ANSA",  # both are wire agencies
    # ── Wikipedia ────────────────────────────────────────────────────────
    "wikipedia.org": "Wikipedia",
    # ── la Repubblica ────────────────────────────────────────────────────
    "repubblica.it": "la Repubblica",
    "la repubblica": "la Repubblica",
    # ── Corriere della Sera ───────────────────────────────────────────────
    "corriere.it": "Corriere della Sera",
    "corriere della sera": "Corriere della Sera",
    # ── Fanpage.it ───────────────────────────────────────────────────────
    "fanpage.it": "Fanpage.it",
    # ── TV2000 ───────────────────────────────────────────────────────────
    "tv2000.it": "TV2000",
    "tv2000it": "TV2000",
    "tg2000": "TV2000",
    # ── Fondazione Umberto Veronesi ──────────────────────────────────────
    "fondazioneveronesi.it": "Fondazione Umberto Veronesi",
    "fondazione veronesi": "Fondazione Umberto Veronesi",
    # ── Avvocato Cittadinanza ────────────────────────────────────────────
    "avvocatocittadinanza.it": "Avvocato Cittadinanza",
    "avvocato cittadinanza": "Avvocato Cittadinanza",
    # ── ISPI ─────────────────────────────────────────────────────────────
    "ispionline.it": "ISPI",
    "ispi - la geopolitica spiegata in modo chiaro": "ISPI",
    # ── Confindustria ────────────────────────────────────────────────────
    "confindustria.it": "Confindustria",
    "confindustria": "Confindustria",
    # ── Corte Costituzionale ─────────────────────────────────────────────
    "cortecostituzionale.it": "Corte Costituzionale",
    "corte costituzionale": "Corte Costituzionale",
    # ── Garante nazionale privati libertà ────────────────────────────────
    "garantenazionaleprivatiliberta.it": "Garante nazionale privati libertà",
    "garante nazionale privati libertà": "Garante nazionale privati libertà",
    # ── LifeGate ─────────────────────────────────────────────────────────
    "lifegate.it": "LifeGate",
    "lifegate": "LifeGate",
    # ── ANCI ─────────────────────────────────────────────────────────────
    "anci.it": "ANCI",
    "anci lombardia": "ANCI",
    # ── Ministero del Lavoro e delle Politiche Sociali ───────────────────
    "lavoro.gov.it": "Ministero del Lavoro e delle Politiche Sociali",
    "ministero del lavoro e delle politiche sociali": "Ministero del Lavoro e delle Politiche Sociali",
    # ── Consulta di Bioetica ─────────────────────────────────────────────
    "consultadibioetica.org": "Consulta di Bioetica",
    "consulta di bioetica": "Consulta di Bioetica",
    # ── Federazione Cure Palliative ──────────────────────────────────────
    "curepalliative.info": "Federazione Cure Palliative",
    "federazione cure palliative": "Federazione Cure Palliative",
    # ── Mondadori Education ──────────────────────────────────────────────
    "mondadorieducation.it": "Mondadori Education",
    "mondadori education": "Mondadori Education",
    # ── La Stampa ────────────────────────────────────────────────────────
    "lastampa.it": "La Stampa",
    "la stampa": "La Stampa",
    # ── Focus ────────────────────────────────────────────────────────────
    "focus.it": "Focus",
}


def resolve_entity(name: str) -> str:
    """
    Map a domain or YouTube channel name to a canonical entity.
    Strategy:
      1. Exact lowercase match
      2. Substring match (handles 'La7 Attualità – Notizie del giorno' etc.)
      3. Fall back to the original name
    """
    if not name:
        return name
    key = name.lower().strip()
    if key in ENTITY_MAP:
        return ENTITY_MAP[key]
    for k, entity in ENTITY_MAP.items():
        if len(k) > 4 and (key.startswith(k) or k in key):
            return entity
    return name


def _parse_json_list(val):
    if not val or isinstance(val, float):
        return []
    try:
        return json.loads(val) if isinstance(val, str) else val
    except Exception:
        return []


print("Entity map loaded:", len(ENTITY_MAP), "entries")
print("resolve_entity examples:")
for ex in [
    "la7.it",
    "La7 Attualità",
    "TG La7",
    "rainews.it",
    "Rai",
    "altalex.com",
    "Altalex News",
    "xyz.it",
]:
    print(f"  {ex!r:35s} → {resolve_entity(ex)!r}")

In [ ]:
# Build lookup: query → set of YouTube-derived entity names
# (uses df_yt from §6e; falls back to empty if not available)
yt_entities_by_query = {}
if "df_yt" in dir() and not df_yt.empty and "channel_label" in df_yt.columns:
    for query, grp in df_yt.groupby("query"):
        yt_entities_by_query[query] = {
            resolve_entity(ch)
            for ch in grp["channel_label"].dropna()
            if ch != "(unknown)"
        }
    print(f"YouTube entity lookup built for {len(yt_entities_by_query)} queries")
else:
    print("WARNING: df_yt not found — run §6e first for full entity overlap.")
    print("         Continuing with domain-only entity resolution.")


def compute_entity_overlap(row):
    """
    Entity-aware overlap:
      AIO entities  = resolved non-YouTube AIO domains
                      + resolved YouTube channel entities
      Organic entities = resolved organic domains
    Returns fraction of AIO entities also in organic entities, or NaN.
    """
    query = row.get("query")

    # ── AIO side ──────────────────────────────────────────────────────
    aio_ents = set()
    for d in _parse_json_list(row.get("aio_domains")):
        if d and "youtube.com" not in d and "youtu.be" not in d:
            aio_ents.add(resolve_entity(d))
    # add YouTube channel entities resolved in §6e
    aio_ents.update(yt_entities_by_query.get(query, set()))

    if not aio_ents:
        return float("nan")

    # ── Organic side ──────────────────────────────────────────────────
    org_ents = {
        resolve_entity(d) for d in _parse_json_list(row.get("organic_domains")) if d
    }

    matched = aio_ents & org_ents
    return len(matched) / len(aio_ents)


df["entity_overlap"] = df.apply(compute_entity_overlap, axis=1)

df_both = df[
    df["has_ai_overview"]
    & df["aio_organic_overlap"].notna()
    & df["entity_overlap"].notna()
].copy()

df_both["overlap_delta"] = df_both["entity_overlap"] - df_both["aio_organic_overlap"]

print(f"Records with both overlap scores: {len(df_both)}")
print()
print(
    pd.DataFrame(
        {
            "Original (domain)": df_both["aio_organic_overlap"],
            "Entity-aware": df_both["entity_overlap"],
            "Delta": df_both["overlap_delta"],
        }
    )
    .describe()
    .round(3)
    .to_string()
)
print()
n_gain = (df_both["overlap_delta"] > 0.001).sum()
print(
    f"Queries where entity-aware overlap > domain overlap: {n_gain} / {len(df_both)} "
    f"({n_gain / len(df_both):.0%})"
)

In [ ]:
# ── 9a. Diagnose the youtube.com spurious-match effect ────────────────────
yt_in_both, yt_aio_only, yt_org_only, no_yt = 0, 0, 0, 0
for _, row in df[df["has_ai_overview"]].iterrows():
    aio_doms = set(_parse_json_list(row.get("aio_domains")))
    org_doms = set(_parse_json_list(row.get("organic_domains")))
    in_aio = "youtube.com" in aio_doms
    in_org = "youtube.com" in org_doms
    if in_aio and in_org:
        yt_in_both += 1
    elif in_aio:
        yt_aio_only += 1
    elif in_org:
        yt_org_only += 1
    else:
        no_yt += 1

n_aio = int(df["has_ai_overview"].sum())
print("youtube.com domain presence in AIO queries:")
print(
    f"  In BOTH aio+organic (spurious match)  : {yt_in_both:3d} / {n_aio}  "
    f"({yt_in_both / n_aio:.0%})  ← domain overlap inflated here"
)
print(f"  In AIO only                           : {yt_aio_only:3d} / {n_aio}")
print(f"  In organic only (no AIO YouTube)      : {yt_org_only:3d} / {n_aio}")
print(f"  No YouTube on either side             : {no_yt:3d} / {n_aio}")
print()
print("The entity-aware metric removes the spurious youtube.com→youtube.com match")
print("and replaces it with channel-level identity (La7, Rai, etc.).")

fig, ax = plt.subplots(figsize=(7, 3.5))
labels = [
    f"YT in both\n(spurious match)\n{yt_in_both} queries",
    f"YT in AIO only\n{yt_aio_only} queries",
    f"YT in organic only\n{yt_org_only} queries",
    f"No YouTube\n{no_yt} queries",
]
values = [yt_in_both, yt_aio_only, yt_org_only, no_yt]
colors = ["#e74c3c", "#e67e22", "#3498db", "#95a5a6"]
bars = ax.bar(labels, values, color=colors, edgecolor="white")
ax.bar_label(bars, padding=3, fontsize=9)
ax.set_ylabel("Number of AIO queries")
ax.set_title(
    "youtube.com domain presence across AIO and organic results\n"
    "Red bar = spurious overlap inflated by domain-level matching",
    fontweight="bold",
)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# ── 9a. Side-by-side distribution ─────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

bins = [i / 20 for i in range(21)]

ax = axes[0]
ax.hist(
    df_both["aio_organic_overlap"],
    bins=bins,
    color="#3498db",
    alpha=0.7,
    label="Original (domain)",
)
ax.hist(
    df_both["entity_overlap"],
    bins=bins,
    color="#e67e22",
    alpha=0.7,
    label="Entity-aware",
)
ax.axvline(
    df_both["aio_organic_overlap"].mean(),
    color="#3498db",
    ls="--",
    lw=1.5,
    label=f"domain mean {df_both['aio_organic_overlap'].mean():.2f}",
)
ax.axvline(
    df_both["entity_overlap"].mean(),
    color="#e67e22",
    ls="--",
    lw=1.5,
    label=f"entity mean {df_both['entity_overlap'].mean():.2f}",
)
ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.set_xlabel("Overlap score")
ax.set_ylabel("Number of queries")
ax.set_title("Overlap distribution\n(domain vs entity-aware)")
ax.legend(fontsize=8)

ax = axes[1]
ax.scatter(
    df_both["aio_organic_overlap"],
    df_both["entity_overlap"],
    alpha=0.35,
    s=18,
    color="#9b59b6",
)
lims = [0, 1]
ax.plot(lims, lims, "k--", lw=0.8, label="y = x  (no change)")
ax.set_xlabel("Original overlap (domains)")
ax.set_ylabel("Entity-aware overlap")
ax.set_title("Original vs entity-aware\n(points above diagonal = gain)")
ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.set_xlim(-0.05, 1.05)
ax.set_ylim(-0.05, 1.05)
ax.legend(fontsize=8)

ax = axes[2]
ax.hist(df_both["overlap_delta"], bins=20, color="#2ecc71", edgecolor="white")
ax.axvline(0, color="black", ls="-", lw=0.8)
ax.axvline(
    df_both["overlap_delta"].mean(),
    color="red",
    ls="--",
    lw=1.3,
    label=f"mean Δ {df_both['overlap_delta'].mean():.3f}",
)
ax.set_xlabel("Δ overlap (entity − domain)")
ax.set_ylabel("Number of queries")
ax.set_title("Overlap gain from entity\nresolution")
ax.legend(fontsize=8)

plt.suptitle(
    "Entity-aware vs domain-level AIO–Organic Overlap", fontweight="bold", y=1.01
)
plt.tight_layout()
plt.show()

### 9b. Cross-platform entity matches

These are the 'new' overlaps the entity-aware metric captures: queries where
the AIO cites a **YouTube channel** whose web domain also appears in the
organic top-10. The original metric would have recorded `youtube.com` as a
non-overlapping domain.

In [ ]:
cp_rows = []
for _, row in df_both.iterrows():
    query = row["query"]
    yt_ents = yt_entities_by_query.get(query, set())
    org_ents = {
        resolve_entity(d) for d in _parse_json_list(row.get("organic_domains")) if d
    }
    for ent in yt_ents & org_ents:
        cp_rows.append(
            {
                "entity": ent,
                "query": query,
                "topic": row["topic"],
                "stance": row["stance"],
                "pro_leaning": row.get("pro_leaning"),
                "orig_overlap": row["aio_organic_overlap"],
                "ent_overlap": row["entity_overlap"],
            }
        )

cp_df = pd.DataFrame(cp_rows)

if cp_df.empty:
    print("No cross-platform matches found (run §6e to enable YouTube resolution).")
else:
    print(f"Cross-platform matches found: {len(cp_df)}")
    print(f"Unique entities matched:      {cp_df['entity'].nunique()}")
    print(f"Unique queries affected:       {cp_df['query'].nunique()}")
    print()

    # Per-entity summary
    ent_summary = (
        cp_df.groupby("entity")
        .agg(
            queries=("query", "nunique"),
            topics=("topic", lambda x: ", ".join(sorted(x.dropna().unique()))),
            stances=("stance", lambda x: ", ".join(sorted(x.dropna().unique()))),
        )
        .sort_values("queries", ascending=False)
    )
    display(ent_summary)

    # Bar chart
    fig, ax = plt.subplots(figsize=(9, max(3, len(ent_summary) * 0.45)))
    ax.barh(
        ent_summary.index[::-1],
        ent_summary["queries"][::-1],
        color="#e74c3c",
        alpha=0.85,
        edgecolor="white",
    )
    for i, (idx, row_s) in enumerate(ent_summary[::-1].iterrows()):
        ax.text(
            row_s["queries"] + 0.05, i, str(row_s["queries"]), va="center", fontsize=9
        )
    ax.set_xlabel("Unique queries where YouTube channel ↔ organic domain")
    ax.set_title(
        "Cross-platform entity matches\n"
        "(AIO cites YouTube channel  |  same brand appears in organic top-10)",
        fontweight="bold",
    )
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.tight_layout()
    plt.show()

### 9c. Entity-aware overlap by topic and stance

In [ ]:
# ── Per topic ──────────────────────────────────────────────────────────────
topics_order = sorted(df_both["topic"].dropna().unique())
fig, axes = plt.subplots(1, 2, figsize=(max(10, len(topics_order) * 2.5), 5))

for ax, col, label, color in [
    (axes[0], "aio_organic_overlap", "Domain overlap", "#3498db"),
    (axes[1], "entity_overlap", "Entity overlap", "#e67e22"),
]:
    sns.boxplot(
        data=df_both,
        x="topic",
        y=col,
        order=topics_order,
        color=color,
        width=0.5,
        ax=ax,
    )
    sns.stripplot(
        data=df_both,
        x="topic",
        y=col,
        order=topics_order,
        color="black",
        size=3,
        alpha=0.4,
        jitter=True,
        ax=ax,
    )
    ax.set_xticklabels(topics_order, rotation=30, ha="right")
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.set_ylabel(label)
    ax.set_xlabel("")
    ax.set_title(f"{label} per topic")
    ax.set_ylim(-0.05, 1.05)

plt.tight_layout()
plt.show()

# ── Per stance ─────────────────────────────────────────────────────────────
df_st = df_both[df_both["stance"].notna()]
stance_order = ["Pro", "Neutral", "Con"]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, col, label in [
    (axes[0], "aio_organic_overlap", "Domain overlap"),
    (axes[1], "entity_overlap", "Entity-aware overlap"),
]:
    sns.boxplot(
        data=df_st,
        x="stance",
        y=col,
        order=stance_order,
        palette=PALETTE,
        width=0.5,
        ax=ax,
    )
    sns.stripplot(
        data=df_st,
        x="stance",
        y=col,
        order=stance_order,
        color="black",
        size=4,
        alpha=0.4,
        jitter=True,
        ax=ax,
    )
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.set_ylabel(label)
    ax.set_xlabel("Query stance")
    ax.set_title(f"{label} by stance")
    ax.set_ylim(-0.05, 1.05)

plt.tight_layout()
plt.show()

# ── Summary table ─────────────────────────────────────────────────────────
summary = pd.DataFrame(
    {
        "domain_mean": df_both.groupby("topic")["aio_organic_overlap"].mean(),
        "entity_mean": df_both.groupby("topic")["entity_overlap"].mean(),
        "mean_delta": df_both.groupby("topic")["overlap_delta"].mean(),
    }
).round(3)
print("Mean overlap by topic:")
display(summary.style.background_gradient(subset=["mean_delta"], cmap="YlGn"))